# Equal/gated fusion: seeds 123 and 2026

Author-supplied runner. The approximate expected-count constant says 7,932,401, but the observed count is 7,932,769, matching the seed-42 source expectation. This stale constant is not evidence of a different architecture. Final run exports are still needed.

Source: `Pasted text.txt`.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
"""
GD-RGBMS-DB — CONTROLLED SIX-CHANNEL ABLATION REPRODUCIBILITY
==============================================================

Runs ONLY seeds 123 and 2026 for the two non-auxiliary six-channel ablation
models from Table 8:
    1) Equal Multi-Scale Fusion Network
    2) Spatially Gated Multi-Scale Fusion Network

Seed 42 is assumed to already exist and is NOT rerun here.

Protocol retained:
- Official GD-RGBMS-DB split: 280 train / 60 validation / 60 test
- Input: RGB + Green + Red-Edge + NIR (6 channels)
- 896 x 672 input size
- AdamW, lr=1e-4, wd=1e-4
- Batch 2, gradient accumulation 2
- Max 70 epochs, early-stop patience 12
- ReduceLROnPlateau factor 0.5, patience 4, min lr 1e-6
- Loss = 0.35 BCE + 0.25 focal + 0.40 Dice
- Residual-scSE encoder-decoder family
- Validation-only checkpoint selection and threshold selection
- Fine threshold search 0.20..0.80 step 0.01
- Final test: no TTA
- num_workers=0 for Colab stability
- Google Drive auto-mount + dataset auto-discovery for Colab

Reference seed-42 values from manuscript Table 8:
- Equal fusion, no auxiliary      : Dice 0.8057 | IoU 0.6746 | P 0.7807 | R 0.8323 | th 0.59
- Spatially gated, no auxiliary   : Dice 0.8062 | IoU 0.6754 | P 0.7858 | R 0.8277 | th 0.55
"""

from __future__ import annotations

import csv
import json
import random
import shutil
import sys
import time
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm import tqdm as _tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

# -----------------------------------------------------------------------------
# COLAB PATH SETTINGS
# -----------------------------------------------------------------------------
# If your dataset is in Google Drive, keep AUTO_MOUNT_DRIVE=True.
# The script will search common locations automatically.
AUTO_MOUNT_DRIVE = True
COPY_FINAL_ZIP_TO_DRIVE = True

# Optional manual dataset path. Leave as None for automatic discovery.
# Example:
# MANUAL_DATASET_DIR = Path("/content/drive/MyDrive/GD-RGBMS-DB")
MANUAL_DATASET_DIR = None

COLAB = False
DRIVE_MOUNTED = False

try:
    from google.colab import drive as _colab_drive
    COLAB = True
except Exception:
    _colab_drive = None

if COLAB and AUTO_MOUNT_DRIVE:
    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.exists():
        print("Mounting Google Drive...")
        _colab_drive.mount("/content/drive")
    DRIVE_MOUNTED = drive_root.exists()

# Colab local SSD is faster for checkpoints/training outputs.
WORK_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
MASTER_OUT_DIR = WORK_DIR / "gd_equal_spatial_gated_seeds123_2026"
MASTER_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Legacy/default candidate kept only as a fallback when the same script is reused elsewhere.
BASE_DIR = Path("/kaggle/input/datasets/hemal7/gd-rgbms-db")

RUN_SEEDS = [123, 2026]
CONFIGS = [
    {
        "key": "equal_fusion",
        "label": "Equal Multi-Scale Fusion Network",
        "threshold_seed42": 0.59,
        "dice_seed42": 0.8057,
    },
    {
        "key": "spatially_gated",
        "label": "Spatially Gated Multi-Scale Fusion Network",
        "threshold_seed42": 0.55,
        "dice_seed42": 0.8062,
    },
]

TARGET_SIZE = (896, 672)  # PIL: width, height
BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
MAX_EPOCHS = 70
PATIENCE = 12
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
SHOW_PROGRESS = False

PREFERRED_GN_GROUPS = 8
BRANCH_BASE = 28
CACHE_TRAIN_AND_VALIDATION = True
MASK_POSITIVE_RGB = np.array([240, 128, 128], dtype=np.int16)

ALL_CHANNEL_NAMES = ["R", "G", "B", "Green", "Red-Edge", "NIR"]

# Same split-level six-channel statistics used in the controlled experiments.
ALL_MEAN = np.array(
    [0.41819795966148376, 0.46573567390441895, 0.2516770660877228,
     0.21822915971279144, 0.16206900775432587, 0.23585167527198792],
    dtype=np.float32,
)
ALL_STD = np.array(
    [0.1540451943874359, 0.1569177359342575, 0.13375557959079742,
     0.07147330790758133, 0.05124884843826294, 0.04640943184494972],
    dtype=np.float32,
)

TRAIN_THRESHOLDS = np.round(np.arange(0.20, 0.851, 0.05), 2).tolist()
FINAL_THRESHOLDS = np.round(np.arange(0.20, 0.801, 0.01), 2).tolist()

EXPECTED_PARAMS = {
    "equal_fusion": 7_932_401,     # approx 7.933 M
    "spatially_gated": 7_950_033,  # deployment model; no auxiliary heads
}

try:
    BILINEAR = Image.Resampling.BILINEAR
    NEAREST = Image.Resampling.NEAREST
except AttributeError:
    BILINEAR = Image.BILINEAR
    NEAREST = Image.NEAREST


def tqdm(*args, **kwargs):
    kwargs["disable"] = not SHOW_PROGRESS
    return _tqdm(*args, **kwargs)


# =============================================================================
# 2. REPRODUCIBILITY / DATASET DISCOVERY
# =============================================================================

def read_list(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def valid_root(path: Path):
    required = [
        path / "RGB",
        path / "MS",
        path / "Mark",
        path / "train_list.txt",
        path / "val_list.txt",
        path / "test_list.txt",
    ]
    if not all(p.exists() for p in required):
        return False
    try:
        return (
            len(read_list(path / "train_list.txt")) == 280
            and len(read_list(path / "val_list.txt")) == 60
            and len(read_list(path / "test_list.txt")) == 60
        )
    except Exception:
        return False


def find_dataset_root():
    # 1) Manual path has highest priority.
    if MANUAL_DATASET_DIR is not None:
        manual = Path(MANUAL_DATASET_DIR)
        if valid_root(manual):
            return manual
        raise FileNotFoundError(
            f"MANUAL_DATASET_DIR was set to {manual}, but it does not contain "
            "RGB/, MS/, Mark/, train_list.txt, val_list.txt, and test_list.txt."
        )

    # 2) Fast checks for common Colab / Drive locations.
    common_candidates = [
        Path("/content/GD-RGBMS-DB"),
        Path("/content/gd-rgbms-db"),
        Path("/content/datasets/GD-RGBMS-DB"),
        Path("/content/datasets/gd-rgbms-db"),
        Path("/content/drive/MyDrive/GD-RGBMS-DB"),
        Path("/content/drive/MyDrive/gd-rgbms-db"),
        Path("/content/drive/MyDrive/datasets/GD-RGBMS-DB"),
        Path("/content/drive/MyDrive/datasets/gd-rgbms-db"),
        BASE_DIR,
    ]
    for candidate in common_candidates:
        if valid_root(candidate):
            return candidate

    # 3) Recursive fallback. Search local /content first, then MyDrive.
    roots = [Path("/content")]
    if DRIVE_MOUNTED:
        roots.append(Path("/content/drive/MyDrive"))
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))

    candidates = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        print(f"Searching for GD-RGBMS-DB under: {root}")
        for train_file in root.rglob("train_list.txt"):
            candidate = train_file.parent
            key = str(candidate.resolve())
            if key in seen:
                continue
            seen.add(key)
            if valid_root(candidate):
                candidates.append(candidate)

    if not candidates:
        raise FileNotFoundError(
            "Could not locate GD-RGBMS-DB. Put the extracted dataset folder in "
            "/content or Google Drive, or set MANUAL_DATASET_DIR near the top of the script. "
            "The dataset root must directly contain RGB/, MS/, Mark/, train_list.txt, "
            "val_list.txt, and test_list.txt."
        )

    candidates.sort(key=lambda p: ("gd-rgbms" not in str(p).lower(), len(str(p))))
    return candidates[0]


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


DATASET_DIR = find_dataset_root()
RGB_DIR = DATASET_DIR / "RGB"
MS_DIR = DATASET_DIR / "MS"
MARK_DIR = DATASET_DIR / "Mark"

TRAIN_NAMES = read_list(DATASET_DIR / "train_list.txt")
VAL_NAMES = read_list(DATASET_DIR / "val_list.txt")
TEST_NAMES = read_list(DATASET_DIR / "test_list.txt")

assert len(TRAIN_NAMES) == 280
assert len(VAL_NAMES) == 60
assert len(TEST_NAMES) == 60
assert not (set(TRAIN_NAMES) & set(VAL_NAMES))
assert not (set(TRAIN_NAMES) & set(TEST_NAMES))
assert not (set(VAL_NAMES) & set(TEST_NAMES))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

print("=" * 100)
print("COLAB | GD-RGBMS-DB — EQUAL FUSION + SPATIALLY GATED | SEEDS 123 AND 2026")
print("=" * 100)
print("Dataset    :", DATASET_DIR)
print("Drive      :", "mounted" if DRIVE_MOUNTED else "not mounted")
print("Device     :", DEVICE)
print("Split      :", len(TRAIN_NAMES), len(VAL_NAMES), len(TEST_NAMES))
print("Input size :", TARGET_SIZE)
print("Models     :", ", ".join([c["label"] for c in CONFIGS]))
print("Seeds      :", RUN_SEEDS)
print("Seed 42    : NOT rerun")
print("=" * 100)


# =============================================================================
# 3. RAW DATA / CACHE
# =============================================================================

def clean_stem(name: str):
    """Return filename stem whether split list contains an extension or not."""
    return Path(str(name).strip()).stem


def find_path(directory: Path, name_or_stem: str, extensions):
    """
    Robust paired-file lookup.

    The official split files may contain RGB names WITH .JPG, while MS/Mark
    files use the same basename with a different extension.  The old version
    appended an extension to the full list entry and therefore searched for
    names such as ``image.JPG.JPG``.  This version first tries the exact name,
    then strips the suffix and searches by basename.
    """
    raw_name = Path(str(name_or_stem).strip()).name

    # 1) Exact filename first (important for RGB entries already ending .JPG).
    exact = directory / raw_name
    if exact.exists() and exact.is_file():
        return exact

    # 2) Strip whatever suffix came from the split list, then try expected ext.
    stem = Path(raw_name).stem
    for ext in extensions:
        p = directory / f"{stem}{ext}"
        if p.exists() and p.is_file():
            return p

    # 3) Case/extension fallback.
    matches = [p for p in directory.glob(f"{stem}.*") if p.is_file()]
    if matches:
        matches.sort()
        return matches[0]

    raise FileNotFoundError(
        f"Could not find paired file for list entry {name_or_stem!r} "
        f"(stem={stem!r}) in {directory}"
    )


def mask_to_binary(image: Image.Image):
    rgb = np.asarray(image.convert("RGB"), dtype=np.int16)
    return (np.abs(rgb - MASK_POSITIVE_RGB).sum(axis=2) < 80).astype(np.float32)


def load_raw(name: str):
    # Split-list entries can include .JPG.  RGB is allowed to match the exact
    # listed filename; MS and mask are paired through the suffix-free basename.
    stem = clean_stem(name)

    rgb_path = find_path(
        RGB_DIR, name,
        [".JPG", ".jpg", ".JPEG", ".jpeg", ".PNG", ".png"],
    )
    ms_path = find_path(
        MS_DIR, stem,
        [".PNG", ".png", ".JPG", ".jpg", ".JPEG", ".jpeg"],
    )
    mask_path = find_path(
        MARK_DIR, stem,
        [".png", ".PNG", ".jpg", ".JPG", ".jpeg", ".JPEG"],
    )

    with Image.open(rgb_path) as im:
        rgb = np.asarray(im.convert("RGB").resize(TARGET_SIZE, BILINEAR), dtype=np.float32) / 255.0
    with Image.open(ms_path) as im:
        ms = np.asarray(im.convert("RGB").resize(TARGET_SIZE, BILINEAR), dtype=np.float32) / 255.0
    with Image.open(mask_path) as im:
        mask = mask_to_binary(im.convert("RGB").resize(TARGET_SIZE, NEAREST))

    image = np.concatenate([rgb, ms], axis=2).astype(np.float32)
    return image, mask, stem


RAW_CACHE = {}
if CACHE_TRAIN_AND_VALIDATION:
    preload_names = sorted(set(TRAIN_NAMES + VAL_NAMES))
    for name in tqdm(preload_names, desc="Preloading train+validation cache"):
        image, mask, stem = load_raw(name)
        RAW_CACHE[name] = (image.astype(np.float16), mask.astype(np.uint8), stem)


def get_raw(name: str):
    cached = RAW_CACHE.get(name)
    if cached is not None:
        image, mask, stem = cached
        return image.astype(np.float32), mask.astype(np.float32), stem
    return load_raw(name)


# =============================================================================
# 4. DATASET / DATALOADER
# =============================================================================

class SixChannelDataset(Dataset):
    def __init__(self, names, augment=False):
        self.names = list(names)
        self.augment = augment
        self.mean = ALL_MEAN.reshape(1, 1, 6)
        self.std = ALL_STD.reshape(1, 1, 6)

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image, mask, stem = get_raw(name)

        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=1).copy()
                mask = np.flip(mask, axis=1).copy()
            if random.random() < 0.5:
                image = np.flip(image, axis=0).copy()
                mask = np.flip(mask, axis=0).copy()
            if random.random() < 0.75:
                gain = np.random.uniform(0.88, 1.12, size=(1, 1, 3)).astype(np.float32)
                bias = np.random.uniform(-0.035, 0.035, size=(1, 1, 3)).astype(np.float32)
                image[..., :3] = np.clip(image[..., :3] * gain + bias, 0.0, 1.0)

        image = ((image - self.mean) / self.std).astype(np.float32)
        return (
            torch.from_numpy(image.transpose(2, 0, 1)).float(),
            torch.from_numpy(mask[None]).float(),
            stem,
        )


def make_loaders(seed: int):
    g = torch.Generator()
    g.manual_seed(seed)
    train_loader = DataLoader(
        SixChannelDataset(TRAIN_NAMES, augment=True),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=g,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )
    val_loader = DataLoader(
        SixChannelDataset(VAL_NAMES, augment=False),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )
    test_loader = DataLoader(
        SixChannelDataset(TEST_NAMES, augment=False),
        batch_size=1,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )
    return train_loader, val_loader, test_loader


# =============================================================================
# 5. MODELS
# =============================================================================

def group_norm(channels: int):
    groups = min(PREFERRED_GN_GROUPS, channels)
    while channels % groups:
        groups -= 1
    return nn.GroupNorm(groups, channels, eps=1e-5, affine=True)


class SCSE(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 1)
        self.channel = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden, 1),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )
        self.spatial = nn.Sequential(nn.Conv2d(channels, 1, 1), nn.Sigmoid())

    def forward(self, x):
        return x * self.channel(x) + x * self.spatial(x)


class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
            nn.SiLU(inplace=True),
            nn.Dropout2d(dropout) if dropout else nn.Identity(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            group_norm(out_channels),
        )
        self.skip = (
            nn.Identity()
            if in_channels == out_channels
            else nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                group_norm(out_channels),
            )
        )
        self.attention = SCSE(out_channels)

    def forward(self, x):
        return self.attention(F.silu(self.block(x) + self.skip(x), inplace=True))


class SoftGatedFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        hidden = max(channels // 8, 8)
        self.gate = nn.Sequential(
            nn.Conv2d(channels * 2, hidden, 1, bias=False),
            group_norm(hidden),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, 2, 1),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            group_norm(channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, rgb_feature, spectral_feature):
        weights = torch.softmax(self.gate(torch.cat([rgb_feature, spectral_feature], dim=1)), dim=1)
        fused = weights[:, 0:1] * rgb_feature + weights[:, 1:2] * spectral_feature
        return self.refine(fused)


class EqualFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.refine = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            group_norm(channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, rgb_feature, spectral_feature):
        fused = 0.5 * (rgb_feature + spectral_feature)
        return self.refine(fused)


class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, dropout=0.0):
        super().__init__()
        self.conv = ResBlock(in_channels + skip_channels, out_channels, dropout)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))


class DualStreamFusionUNet(nn.Module):
    def __init__(self, fusion_type="equal", base=BRANCH_BASE):
        super().__init__()
        c1, c2, c3, c4 = base, base * 2, base * 4, base * 8
        self.pool = nn.MaxPool2d(2)

        self.rgb1 = ResBlock(3, c1)
        self.spec1 = ResBlock(3, c1)
        self.rgb2 = ResBlock(c1, c2)
        self.spec2 = ResBlock(c1, c2)
        self.rgb3 = ResBlock(c2, c3, 0.05)
        self.spec3 = ResBlock(c2, c3, 0.05)
        self.rgb4 = ResBlock(c3, c4, 0.10)
        self.spec4 = ResBlock(c3, c4, 0.10)

        if fusion_type == "equal":
            Fusion = EqualFusion
        elif fusion_type == "spatially_gated":
            Fusion = SoftGatedFusion
        else:
            raise ValueError(f"Unknown fusion_type: {fusion_type}")

        self.fuse1 = Fusion(c1)
        self.fuse2 = Fusion(c2)
        self.fuse3 = Fusion(c3)
        self.fuse4 = Fusion(c4)

        self.center = ResBlock(c4, c4 * 2, 0.10)
        self.up4 = UpBlock(c4 * 2, c4, c4, 0.10)
        self.up3 = UpBlock(c4, c3, c3, 0.05)
        self.up2 = UpBlock(c3, c2, c2)
        self.up1 = UpBlock(c2, c1, c1)
        self.output = nn.Conv2d(c1, 1, 1)

    def forward(self, x):
        if x.ndim != 4 or x.shape[1] != 6:
            raise ValueError(f"Expected [B,6,H,W], received {tuple(x.shape)}")

        rgb = x[:, :3]
        spectral = x[:, 3:]

        r1 = self.rgb1(rgb)
        s1 = self.spec1(spectral)
        f1 = self.fuse1(r1, s1)

        r2 = self.rgb2(self.pool(r1 + f1))
        s2 = self.spec2(self.pool(s1 + f1))
        f2 = self.fuse2(r2, s2)

        r3 = self.rgb3(self.pool(r2 + f2))
        s3 = self.spec3(self.pool(s2 + f2))
        f3 = self.fuse3(r3, s3)

        r4 = self.rgb4(self.pool(r3 + f3))
        s4 = self.spec4(self.pool(s3 + f3))
        f4 = self.fuse4(r4, s4)

        center = self.center(self.pool(f4))
        d4 = self.up4(center, f4)
        d3 = self.up3(d4, f3)
        d2 = self.up2(d3, f2)
        d1 = self.up1(d2, f1)
        return self.output(d1)


def build_model(config_key: str):
    fusion_type = "equal" if config_key == "equal_fusion" else "spatially_gated"
    model = DualStreamFusionUNet(fusion_type=fusion_type).to(DEVICE)
    total = sum(p.numel() for p in model.parameters())
    print(f"Parameters ({config_key}): {total:,}")

    expected = EXPECTED_PARAMS.get(config_key)
    if expected is not None:
        diff = total - expected
        print(f"Expected parameter count approx: {expected:,} | diff: {diff:+,}")
    return model


# =============================================================================
# 6. LOSS / METRICS
# =============================================================================

class ComboLoss(nn.Module):
    def forward(self, logits, targets):
        targets = targets.float()
        bce_map = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        bce = bce_map.mean()
        probs = torch.sigmoid(logits)
        pt = probs * targets + (1.0 - probs) * (1.0 - targets)
        focal = (bce_map * ((1.0 - pt) ** 2.0)).mean()
        dims = tuple(range(1, logits.ndim))
        intersection = (probs * targets).sum(dims)
        denominator = probs.sum(dims) + targets.sum(dims)
        dice_loss = 1.0 - ((2.0 * intersection + 1e-6) / (denominator + 1e-6)).mean()
        return 0.35 * bce + 0.25 * focal + 0.40 * dice_loss


criterion = ComboLoss()


def confusion_numpy(probability, mask, threshold):
    pred = probability >= threshold
    target = mask >= 0.5
    return [
        int(np.logical_and(pred, target).sum()),
        int(np.logical_and(pred, ~target).sum()),
        int(np.logical_and(~pred, target).sum()),
        int(np.logical_and(~pred, ~target).sum()),
    ]


def confusion_tensor(probability, mask, threshold):
    pred = probability >= threshold
    target = mask >= 0.5
    return [
        int((pred & target).sum().item()),
        int((pred & ~target).sum().item()),
        int((~pred & target).sum().item()),
        int((~pred & ~target).sum().item()),
    ]


def add_confusion(a, b):
    return [int(x + y) for x, y in zip(a, b)]


def metric_dict(values):
    tp, fp, fn, tn = values
    e = 1e-7
    return {
        "dice": (2 * tp + e) / (2 * tp + fp + fn + e),
        "iou": (tp + e) / (tp + fp + fn + e),
        "precision": (tp + e) / (tp + fp + e),
        "recall": (tp + e) / (tp + fn + e),
        "accuracy": (tp + tn + e) / (tp + fp + fn + tn + e),
    }


def save_csv(path: Path, rows):
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


# =============================================================================
# 7. EVALUATION HELPERS
# =============================================================================

@torch.no_grad()
def collect_records(loader, model, desc):
    model.eval()
    records = []
    for images, masks, stems in tqdm(loader, desc=desc, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            probs = torch.sigmoid(model(images))
        records.append({
            "stem": stems[0],
            "probability": probs[0, 0].float().cpu().numpy(),
            "mask": masks[0, 0].float().numpy(),
        })
    return records


def search_threshold(records, thresholds):
    rows = []
    for threshold in thresholds:
        total = [0, 0, 0, 0]
        for record in records:
            total = add_confusion(total, confusion_numpy(record["probability"], record["mask"], threshold))
        row = metric_dict(total)
        row["threshold"] = float(threshold)
        rows.append(row)
    best = max(rows, key=lambda row: row["dice"])
    return best, rows


# =============================================================================
# 8. MAIN TRAIN/EVAL PER RUN
# =============================================================================

def run_one(config, seed):
    config_key = config["key"]
    config_label = config["label"]
    run_name = f"{config_key}_seed{seed}"
    run_dir = MASTER_OUT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 100)
    print(f"RUN: {config_label} | seed {seed}")
    print("#" * 100)

    seed_everything(seed)
    train_loader, val_loader, test_loader = make_loaders(seed)

    model = build_model(config_key)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=4, min_lr=1e-6
    )
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

    checkpoint_path = run_dir / "best_checkpoint.pt"
    history_rows = []
    best_val_dice = -1.0
    best_epoch = 0

    def train_epoch(epoch):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        for step, (images, masks, _) in enumerate(tqdm(train_loader, desc=f"{config_key} seed {seed} train {epoch}", leave=False), start=1):
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, masks)
                scaled_loss = loss / ACCUMULATION_STEPS
            scaler.scale(scaled_loss).backward()
            if step % ACCUMULATION_STEPS == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            total_loss += loss.item()
        return total_loss / max(len(train_loader), 1)

    @torch.no_grad()
    def validate_checkpoint():
        model.eval()
        threshold_totals = {th: [0, 0, 0, 0] for th in TRAIN_THRESHOLDS}
        loss_sum = 0.0
        for images, masks, _ in tqdm(val_loader, desc=f"{config_key} seed {seed} validation", leave=False):
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, masks)
                probs = torch.sigmoid(logits)
            loss_sum += loss.item()
            for th in TRAIN_THRESHOLDS:
                threshold_totals[th] = add_confusion(threshold_totals[th], confusion_tensor(probs, masks, th))
        rows = []
        for th in TRAIN_THRESHOLDS:
            row = metric_dict(threshold_totals[th])
            row["threshold"] = float(th)
            rows.append(row)
        best = max(rows, key=lambda row: row["dice"])
        best["loss"] = loss_sum / max(len(val_loader), 1)
        return best

    for epoch in range(1, MAX_EPOCHS + 1):
        start = time.time()
        current_lr = float(optimizer.param_groups[0]["lr"])
        train_loss = train_epoch(epoch)
        validation = validate_checkpoint()
        scheduler.step(validation["dice"])

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": validation["loss"],
            "val_dice": validation["dice"],
            "val_iou": validation["iou"],
            "val_precision": validation["precision"],
            "val_recall": validation["recall"],
            "val_accuracy": validation["accuracy"],
            "val_threshold": validation["threshold"],
            "learning_rate": current_lr,
            "minutes": (time.time() - start) / 60.0,
        }
        history_rows.append(row)

        print(
            f"{config_key} seed {seed} | epoch {epoch:03d} | loss {train_loss:.4f} | "
            f"val Dice {validation['dice']:.4f} | IoU {validation['iou']:.4f} | "
            f"P {validation['precision']:.4f} | R {validation['recall']:.4f} | "
            f"th {validation['threshold']:.2f} | lr {current_lr:.2e}"
        )

        if validation["dice"] > best_val_dice:
            best_val_dice = float(validation["dice"])
            best_epoch = epoch
            torch.save({
                "model": model.state_dict(),
                "best_epoch": best_epoch,
                "best_validation_dice": best_val_dice,
                "coarse_validation_threshold": validation["threshold"],
                "seed": seed,
                "config_key": config_key,
            }, checkpoint_path)

        if epoch - best_epoch >= PATIENCE:
            print(f"Early stopping after {PATIENCE} non-improving epochs.")
            break

    save_csv(run_dir / "training_history.csv", history_rows)

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model"])

    val_records = collect_records(val_loader, model, f"{config_key} seed {seed} fine threshold validation")
    val_best, val_threshold_rows = search_threshold(val_records, FINAL_THRESHOLDS)
    selected_threshold = float(val_best["threshold"])
    save_csv(run_dir / "validation_threshold_search.csv", val_threshold_rows)

    test_records = collect_records(test_loader, model, f"{config_key} seed {seed} final test")
    total = [0, 0, 0, 0]
    per_image_rows = []
    for record in test_records:
        values = confusion_numpy(record["probability"], record["mask"], selected_threshold)
        total = add_confusion(total, values)
        row = metric_dict(values)
        row.update({
            "stem": record["stem"],
            "threshold": selected_threshold,
            "tp": values[0],
            "fp": values[1],
            "fn": values[2],
            "tn": values[3],
        })
        per_image_rows.append(row)
    test_result = metric_dict(total)
    test_result.update({
        "config_key": config_key,
        "config_label": config_label,
        "seed": seed,
        "train_count": len(TRAIN_NAMES),
        "validation_count": len(VAL_NAMES),
        "test_count": len(TEST_NAMES),
        "best_epoch": int(checkpoint["best_epoch"]),
        "best_validation_dice": float(checkpoint["best_validation_dice"]),
        "coarse_validation_threshold": float(checkpoint["coarse_validation_threshold"]),
        "fine_validation_dice": float(val_best["dice"]),
        "selected_threshold": selected_threshold,
        "tp": total[0],
        "fp": total[1],
        "fn": total[2],
        "tn": total[3],
    })

    save_csv(run_dir / "final_test_metrics.csv", [test_result])
    save_csv(run_dir / "per_image_test_metrics.csv", per_image_rows)

    with open(run_dir / "experiment_configuration.json", "w", encoding="utf-8") as f:
        json.dump({
            "dataset": str(DATASET_DIR),
            "config_key": config_key,
            "config_label": config_label,
            "seed": seed,
            "split_counts": {"train": len(TRAIN_NAMES), "validation": len(VAL_NAMES), "test": len(TEST_NAMES)},
            "target_size": list(TARGET_SIZE),
            "batch_size": BATCH_SIZE,
            "accumulation_steps": ACCUMULATION_STEPS,
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "optimizer": "AdamW",
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "channels": ALL_CHANNEL_NAMES,
            "normalization_mean": ALL_MEAN.tolist(),
            "normalization_std": ALL_STD.tolist(),
            "train_time_coarse_thresholds": TRAIN_THRESHOLDS,
            "fine_thresholds": FINAL_THRESHOLDS,
            "reference_seed42": {
                "dice": config.get("dice_seed42"),
                "threshold": config.get("threshold_seed42"),
            },
            "test_metrics": test_result,
        }, f, indent=2)

    print("-" * 80)
    print(config_label, f"| seed {seed}")
    print("Best epoch           :", checkpoint["best_epoch"])
    print("Best validation Dice :", f"{checkpoint['best_validation_dice']:.4f}")
    print("Selected threshold   :", f"{selected_threshold:.2f}")
    print("Test Dice            :", f"{test_result['dice']:.4f}")
    print("Test IoU             :", f"{test_result['iou']:.4f}")
    print("Test Precision       :", f"{test_result['precision']:.4f}")
    print("Test Recall          :", f"{test_result['recall']:.4f}")
    print("Test Accuracy        :", f"{test_result['accuracy']:.4f}")
    print("Output dir           :", run_dir)

    return test_result


# =============================================================================
# 9. EXECUTE ALL REQUESTED RUNS
# =============================================================================

all_results = []

for config in CONFIGS:
    for seed in RUN_SEEDS:
        all_results.append(run_one(config, seed))

save_csv(MASTER_OUT_DIR / "all_results.csv", all_results)

# Group summary by config.
summary_rows = []
for config in CONFIGS:
    rows = [r for r in all_results if r["config_key"] == config["key"]]
    if not rows:
        continue
    for metric in ["dice", "iou", "precision", "recall", "accuracy", "fine_validation_dice"]:
        values = np.array([r[metric] for r in rows], dtype=np.float64)
        config[f"mean_{metric}"] = float(values.mean())
        config[f"sd_{metric}"] = float(values.std(ddof=1)) if len(values) > 1 else 0.0
    config["n_seeds_new"] = len(rows)
    summary_rows.append({
        "config_key": config["key"],
        "config_label": config["label"],
        "n_new_seeds": len(rows),
        "seed42_reference_dice": config.get("dice_seed42"),
        "mean_test_dice_new_seeds": config["mean_dice"],
        "sd_test_dice_new_seeds": config["sd_dice"],
        "mean_test_iou_new_seeds": config["mean_iou"],
        "sd_test_iou_new_seeds": config["sd_iou"],
        "mean_precision_new_seeds": config["mean_precision"],
        "sd_precision_new_seeds": config["sd_precision"],
        "mean_recall_new_seeds": config["mean_recall"],
        "sd_recall_new_seeds": config["sd_recall"],
        "mean_threshold_new_seeds": float(np.mean([r["selected_threshold"] for r in rows])),
        "sd_threshold_new_seeds": float(np.std([r["selected_threshold"] for r in rows], ddof=1)) if len(rows) > 1 else 0.0,
    })

save_csv(MASTER_OUT_DIR / "summary_new_seeds_only.csv", summary_rows)

archive = shutil.make_archive(
    str(WORK_DIR / "gd_equal_spatial_gated_seeds123_2026_complete"),
    "zip",
    str(MASTER_OUT_DIR),
)

# Copy one final ZIP to MyDrive so the results survive a Colab runtime reset.
drive_archive = None
if COPY_FINAL_ZIP_TO_DRIVE and DRIVE_MOUNTED:
    backup_dir = Path("/content/drive/MyDrive/GD_multiseed_results")
    backup_dir.mkdir(parents=True, exist_ok=True)
    drive_archive = backup_dir / Path(archive).name
    shutil.copy2(archive, drive_archive)

print("\n" + "=" * 100)
print("ALL RUNS FINISHED")
print("=" * 100)
for row in all_results:
    print(
        f"{row['config_key']} | seed {row['seed']} | "
        f"Dice={row['dice']:.4f} | IoU={row['iou']:.4f} | "
        f"P={row['precision']:.4f} | R={row['recall']:.4f} | th={row['selected_threshold']:.2f}"
    )
print("\nLocal output :", MASTER_OUT_DIR)
print("Local ZIP    :", archive)
if drive_archive is not None:
    print("Drive ZIP    :", drive_archive)
print("Done.")
